# **Búsqueda Primero en Profundidad - Depth-First Search (DFS)**

- Busqueda primero en profundidad → Depth-First Search (DFS)

- Siempre expande el nodo más profundo en la frontera actual del árbol de búsqueda. Cuando esos nodos se expanden, son quitados de la frontera, así entonces la búsqueda “retrocede” al siguiente nodo más superficial.
    
    <aside>
    💡
    
    El algoritmo solo “retrocede” si tiene una profundidad maxima. Si el arbol es infinito, nunca va a parar de meterse en profundidad
    
    </aside>
    
- No encuentra la solución más eficiente, pero consume muy poca memoria $O(b \cdot m)$, y el tiempo es proporcional a la cantidad de estados. Siendo
    - $b$ la cantidad de branches que se abren en cada nodo
    - $m$ la profundidad en donde esta la solucion
    
    <aside>
    💡
    
    La solucion toma un tiempo y memoria computacional de orden lineal
    
    </aside>

- Usamos una cola LIFO donde vamos agregando los nodo frontera

    <aside>
    💡

    Al agregar nodos en la cola siempre agrego de izquierda a derecha. La diferencia esta en el orden en que saco los nodos (LIFO)

    </aside>

**Detalles de implementacion**

La función debe devolver la salida correspondiente a la solución encontrada o `None` si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [2]:
# Búsqueda Primero en Profundidad

def deapth_first_search(number_disks=5) -> (NodeHanoi, dict):
    # Inicializamos el problema
    list_disks = [i for i in range(5, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    frontier = [NodeHanoi(problem.initial)] # Cola LIFO con el nodo inicial
    explored = set() # Conjunto de estados ya visitados

    node_explored = 0
    
    while len(frontier) != 0:
        node = frontier.pop()
        node_explored += 1
        
        explored.add(node.state) # Verificamos si llegamos al objetivo
        
        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": node.depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics
        
        # Agregamos a la frontera los nodos sucesores que no hayan sido visitados
        for next_node in node.expand(problem):
            if next_node.state not in explored:
                frontier.append(next_node)

    # Si no se encuentra solución, devolvemos métricas igualmente
    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": node.depth, # OBS: Si no se encontró la solución, este valor solo tiene sentido en breadth_first_search, en otros casos se debe ir llevando registro de cual fue la máxima profundidad
        "cost_total": None,
    }
    return None, metrics

Se prueba la función:

In [3]:
solution, metrics = deapth_first_search(number_disks=5)

Veamos las métricas:

In [4]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 122
states_visited: 122
nodes_in_frontier: 63
max_depth: 121
cost_total: 121.0


Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 2 | 1 | >
<Node HanoiState: 5 4 3 | 1 | 2>
<Node HanoiState: 5 4 3 |  | 2 1>
<Node HanoiState: 5 4 3 1 |  | 2>
<Node HanoiState: 5 4 3 1 | 2 | >
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 | 2 | 3 1>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 4 | 1 | 3 2>
<Node HanoiState: 5 4 2 | 1 | 3>
<Node HanoiState: 5 4 2 |  | 3 1>
<Node HanoiState: 5 4 2 1 |  | 3>
<Node HanoiState: 5 4 2 1 | 3 | >
<Node HanoiState: 5 4 2 | 3 | 1>
<Node HanoiState: 5 4 2 | 3 1 | >
<Node HanoiState: 5 4 | 3 1 | 2>
<Node HanoiState: 5 4 | 3 | 2 1>
<Node HanoiState: 5 4 1 | 3 | 2>
<Node HanoiState: 5 4 1 | 3 2 | >
<Node HanoiState: 5 4 | 3 2 | 1>
<Node HanoiState: 5 4 | 3 2 1 | >
<Node HanoiState: 5 | 3 2 1 | 4>
<Node HanoiState: 5 | 3 2 | 4 1>
<Node HanoiState: 5 1 | 3 2

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 3 to 2
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 3 to 2
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 4 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 3 from 2 to 3
Move disk 1 from 1 to 3
Move disk 1 from 3 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 1 from 3 to 1
Move disk 2 from

# Simulacion

In [7]:
solution.generate_solution_for_simulator()

Mover los archivos `initial_state.json` & `sequence.json` dentro de la carpeta `simulator/` y ejecutar

```
python3 ./simulation_hanoi.py
```